# Controlled Decoding: шаг 6, CD против best-of-K при точно равном числе токенов

В `03`–`05` CD сравнивался с ближайшим best-of-K, у которого токенов **не меньше**: K = 2, 4, 8, 16, 32. Если CD
сгенерировал чуть больше, чем best-of-4, его сравнивали с best-of-8, у которого токенов почти вдвое больше. Такое
сравнение занижает CD.

Здесь best-of-K считается для **любого целого K** от 1 до 32 и интерполируется до дробного K* = (токены CD) / (средняя
длина ответа π_ref), то есть до точно того же бюджета токенов.

Reward best-of-K при целом K считается точно: ожидаемый максимум reward по случайному подмножеству из K сэмплов среди 32
имеющихся. Если reward 32 сэмплов отсортированы по возрастанию r₁ ≤ … ≤ r₃₂, то
E[best-of-K] = Σᵢ rᵢ · C(i−1, K−1) / C(32, K).
Это тот же best-of-K, что и раньше, но без случайного разбиения на группы, поэтому шум меньше.

Сравнение парное, бутстреп по промптам. GPU не нужен: **Accelerator = None**.

**Входы (Add Input):** `cd-hh-eval` (T=1), `cd-hh-feedback` (T=0.7, старый scorer), `cd-hh-t07` (T=0.7, новый scorer).

In [ ]:
import os, re, glob, json, math
import numpy as np
import pandas as pd

N_BOOT = 5000
KMAX = 32

def find_dir(pattern, ok=lambda d: True):
    found = sorted({os.path.dirname(f) for f in glob.glob(f"/kaggle/input/**/{pattern}", recursive=True)
                    if "debug" not in f and ok(os.path.dirname(f))})
    assert found, f"не найдено: {pattern}"
    return found[0]

EVAL = find_dir("bon_000.parquet", lambda d: json.load(open(f"{d}/run_meta.json"))["n_prompts"] == 500)
FB = find_dir("tbon_000.parquet")
T07 = find_dir("t07_meta.json")
print("T=1:", EVAL, "\nT=0.7 старый scorer:", FB, "\nT=0.7 новый scorer:", T07)

def load(d, name):
    df = pd.concat([pd.read_parquet(f) for f in sorted(glob.glob(f"{d}/{name}_[0-9][0-9][0-9].parquet"))], ignore_index=True)
    return df.sort_values(["pid", "sample"] if "sample" in df else "pid").reset_index(drop=True)

## Best-of-K для любого целого K

In [ ]:
def bon_exact(R):
    # R: (n, 32) reward сэмплов → (n, 32): E[best-of-K] для K = 1..32
    n, m = R.shape
    S = np.sort(R, 1)
    i = np.arange(1, m + 1)
    W = np.array([[math.comb(ii - 1, K - 1) / math.comb(m, K) for ii in i] for K in range(1, m + 1)])  # (K, i)
    return S @ W.T

def at_k(curve, k):
    # curve: (..., 32) для K = 1..32; линейная интерполяция по K до дробного k
    k = np.clip(k, 1, KMAX)
    lo = int(np.floor(k)); hi = min(lo + 1, KMAX); w = k - lo
    return (1 - w) * curve[..., lo - 1] + w * curve[..., hi - 1]

def k_eq(c, curve_mean):
    # дробный K, при котором best-of-K даёт reward c (кривая монотонна по построению)
    Ks = np.arange(1, KMAX + 1)
    return float(np.interp(c, np.maximum.accumulate(curve_mean), Ks))

def ci(x):
    lo, hi = np.percentile(x, [2.5, 97.5])
    return f"[{lo:.2f}; {hi:.2f}]"

# Проверка: точная формула совпадает с прямым перебором на маленьком примере
_r = np.random.default_rng(0).normal(size=(1, 6))
from itertools import combinations
_brute = [np.mean([max(c) for c in combinations(_r[0], K)]) for K in range(1, 7)]
_S = np.sort(_r, 1); _W = np.array([[math.comb(i - 1, K - 1) / math.comb(6, K) for i in range(1, 7)] for K in range(1, 7)])
assert np.allclose(_S @ _W.T, _brute)

## Сравнение

In [ ]:
def compare_set(label, bon, cds):
    pids = np.sort(bon.pid.unique())
    bon = bon[bon["sample"] < KMAX]
    R = bon.reward.values.reshape(len(pids), KMAX)
    curve = bon_exact(R)                       # (n, 32)
    L = bon.response_ids.map(len).mean()       # средняя длина ответа π_ref: цена одного сэмпла
    boot = np.random.default_rng(0).integers(0, len(pids), (N_BOOT, len(pids)))
    rows = []
    for name, d in cds.items():
        assert (d.pid.values == pids).all(), name
        c = d.reward.values
        k_tok = d.gen_tokens.mean() / L        # K* при том же бюджете токенов
        b = at_k(curve, k_tok)                 # (n,)
        diff = (c - b)[boot].mean(1)
        ke = k_eq(c.mean(), curve.mean(0))
        ke_b = [k_eq(c[i].mean(), curve[i].mean(0)) for i in boot[:1000]]  # 1000 повторов: K_eq дорогой
        rows.append({"set": label, "method": name, "reward": c.mean(), "gen_tok": d.gen_tokens.mean(),
                     "K при равных токенах": k_tok, "best-of-K* reward": b.mean(),
                     "Δ": c.mean() - b.mean(), "Δ ci": ci(diff), "P(Δ≤0)": float((diff <= 0).mean()),
                     "K_eq": ke, "K_eq ci": ci(ke_b), "K_eq / K": ke / k_tok})
    return pd.DataFrame(rows), curve.mean(0), L

def cd_sort(names):
    return sorted(names, key=lambda s: tuple(map(int, re.findall(r"\d+", s)))[::-1])

def cd_names(d, prefix):
    return cd_sort({re.sub(r"_\d+\.parquet$", "", os.path.basename(f)) for f in glob.glob(f"{d}/{prefix}_M*_K*_*.parquet")})

def pretty(name):
    M, K = re.findall(r"\d+", name)
    return f"CD M={M} K={K}"

bon1 = load(EVAL, "bon")
bon7 = load(FB, "tbon")
res1, curve1, L1 = compare_set("T=1", bon1, {pretty(n): load(EVAL, n) for n in cd_names(EVAL, "cd")})
res7o, curve7, L7 = compare_set("T=0.7, scorer T=1", bon7, {pretty(n): load(FB, n) for n in cd_names(FB, "tcd")})
res7n, _, _ = compare_set("T=0.7, scorer T=0.7", bon7, {pretty(n): load(T07, n) for n in cd_names(T07, "ncd")})

print(f"средняя длина ответа π_ref: T=1 {L1:.1f}, T=0.7 {L7:.1f}")
print("best-of-K, T=1:  ", {K: round(curve1[K - 1], 2) for K in [1, 2, 4, 8, 16, 32]})
print("best-of-K, T=0.7:", {K: round(curve7[K - 1], 2) for K in [1, 2, 4, 8, 16, 32]})

out = pd.concat([res1, res7o, res7n], ignore_index=True)
out.to_csv("equal_tokens.csv", index=False)
pd.set_option("display.width", 250)
print("\nΔ = reward CD − reward best-of-K* при точно том же числе токенов; K_eq / K > 1 значит, что CD эффективнее best-of-K")
print(out.round(2).to_string(index=False))